This notebook is for generating "silver label" examples using the trained span identification and technique classification models in order to train a lighter weight model. The raw news article data pre-adding silver labels is from the English-only subset of the Common Crawl News dataset. Once run through the existing models to get "silver labels," we use these examples to train a xx model to be used in our Chrome extension.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path
from datasets import load_dataset
import os
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification
import json
from tqdm.auto import tqdm


In [2]:
#Identify base directory to ensure portability
BASE_DIR = Path.cwd().resolve().parent
MODELS_DIR = BASE_DIR / "models"
interim_dir = BASE_DIR / "data" / "interim"
interim_dir.mkdir(parents=True, exist_ok=True)
output_file = interim_dir / "news_with_labels.csv"
DATA_PATH = BASE_DIR / "data" / "processed" / "semeval_tc_cleaned.csv"

SI_DIR = MODELS_DIR / "semeval_roberta_scanner"
SI_SPEC_DIR = MODELS_DIR / "semeval_roberta_scanner_specialist"
TC_DIR = MODELS_DIR / "semeval_roberta_classifier"

SI_MODEL_PATH = f"{os.fspath(SI_DIR.absolute())}"
SI_SPEC_PATH = f"{os.fspath(SI_SPEC_DIR.absolute())}"
TC_MODEL_PATH = f"{os.fspath(TC_DIR.absolute())}"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [3]:
#Run training notebooks if models are missing
REQUIRED_FILES = ["config.json", "model.safetensors"]

def model_exists(path):
    path = Path(path)
    has_weights = any(path.glob("*.bin")) or any(path.glob("*.safetensors"))
    return has_weights

if not model_exists(SI_MODEL_PATH):
    print("SI Model missing. Running training notebook...")
    %run 4.1-fp-semeval-si-modeling.ipynb
if not model_exists(TC_MODEL_PATH):
    print("TC Model missing. Running training notebook...")
    %run 4.2-fp-semeval-tc-modeling.ipynb

In [4]:
#Load Base SI Model (RoBERTa token-classifier for span detection)
print(f"Loading Base SI Model from: {SI_MODEL_PATH}...")
si_tokenizer = AutoTokenizer.from_pretrained(SI_MODEL_PATH)
si_model = AutoModelForTokenClassification.from_pretrained(SI_MODEL_PATH, local_files_only=True).to(device)
si_model.eval()

#Load Specialist SI Model
print(f"Loading Specialist SI Model from: {SI_SPEC_PATH}...")
si_spec_model = AutoModelForTokenClassification.from_pretrained(SI_SPEC_PATH, local_files_only=True).to(device)
si_spec_model.eval()

Loading Base SI Model from: /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/semeval_roberta_scanner...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading Specialist SI Model from: /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/semeval_roberta_scanner_specialist...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForTokenClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (L

In [5]:
#Load TC Model (Technique Classification)
tc_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
tc_model = AutoModelForSequenceClassification.from_pretrained(TC_MODEL_PATH).to(device)
tc_model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50267, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.2, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [6]:
#Set thresholds for each technique
OPTIMIZED_THRESHOLDS = {
    'Appeal_to_Authority': 0.60,
    'Appeal_to_fear-prejudice': 0.50,
    'Bandwagon_Reductio_ad_hitlerum': 0.10,
    'Black-and-White_Fallacy': 0.20,
    'Causal_Oversimplification': 0.20,
    'Doubt': 0.35,
    'Exaggeration_Minimisation': 0.40,
    'Flag-Waving': 0.45,
    'Loaded_Language': 0.40,
    'Name_Calling_Labeling': 0.55,
    'Repetition': 0.40,
    'Slogans': 0.30,
    'Thought-terminating_Cliches': 0.15,
    'Whataboutism_Straw_Men_Red_Herring': 0.15
}

In [7]:
def run_pipeline_batched(texts):
    """
    Processes a list of texts through the SI -> Cascade -> TC pipeline.
    """
    #1. SI Tokenization (Batch)
    inputs = si_tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        return_offsets_mapping=True
    ).to(device)

    offsets_batch = inputs.pop("offset_mapping")

    with torch.no_grad():
        #2. SI Model Inference (Batch)
        base_outputs = si_model(**inputs)
        base_preds_batch = torch.argmax(base_outputs.logits, dim=-1)

        # 3. Specialist Model Inference (Batch)
        spec_outputs = si_spec_model(**inputs)
        spec_probs_batch = F.softmax(spec_outputs.logits, dim=-1)
        propaganda_prob_batch = spec_probs_batch[:, :, 1]

    #4. Apply Cascade Logic & Extract Spans for each item in batch
    batch_final_results = []

    for i in range(len(texts)):
        text = texts[i]
        base_preds = base_preds_batch[i]
        prop_probs = propaganda_prob_batch[i]
        offsets = offsets_batch[i]

        #Merge predictions
        final_preds = base_preds.clone()
        mask = (base_preds == 0) & (prop_probs > 0.5)
        final_preds[mask] = 1

        #Extract spans
        predicted_spans = []
        current_span = None
        for j, pred in enumerate(final_preds):
            label = pred.item()
            start, end = offsets[j]
            if start == end: continue
            if label in [1, 2]:
                if current_span is None:
                    current_span = [start.item(), end.item()]
                else:
                    current_span[1] = end.item()
            elif current_span:
                predicted_spans.append(tuple(current_span))
                current_span = None
        if current_span: predicted_spans.append(tuple(current_span))

        #5. Technique Classification (TC) for extracted spans
        article_results = []
        for span in predicted_spans:
            span_text = text[span[0]:span[1]].strip()
            if not span_text: continue

            tc_inputs = tc_tokenizer(span_text, return_tensors="pt", truncation=True, padding=True).to(device)
            with torch.no_grad():
                tc_logits = tc_model(**tc_inputs).logits
                probs = torch.sigmoid(tc_logits)[0]

            found_techniques = []
            for class_id, prob in enumerate(probs):
                tech_name = tc_model.config.id2label[class_id]
                if prob.item() >= OPTIMIZED_THRESHOLDS.get(tech_name, 0.5):
                    found_techniques.append(tech_name)

            if not found_techniques:
                found_techniques.append(tc_model.config.id2label[torch.argmax(probs).item()])

            for tech in found_techniques:
                article_results.append({"span": tuple(span), "technique": tech})

        batch_final_results.append(article_results)

    return batch_final_results

In [ ]:
BATCH_SIZE = 32

#1. Load or Initialize the Dataframe
if not output_file.exists():
    print("No cache found. Preparing dataset...")
    news = load_dataset("vblagoje/cc_news", split="train")
    news = news.to_pandas()

    #Constrain to only the text, as that's the only input our extension will be given
    #And take a random sample of articles because the dataset is unreasonably large
    news = news[['text']].sample(frac=0.1, random_state=42).reset_index(drop=True)
    news.iloc[0:0].to_csv(output_file, index=False)
    processed_count = 0
else:
    #Check how many rows we've already done
    existing_data = pd.read_csv(output_file)
    processed_count = len(existing_data)
    print(f"Resuming from article {processed_count}...")

    #We still need the 'news' dataframe to know what's left to process
    #Re-fetch the same sampled dataset
    news = load_dataset("vblagoje/cc_news", split="train").to_pandas()
    news = news[['text']].sample(frac=0.1, random_state=42).reset_index(drop=True)

#2. The Batch Processing Loop
texts_to_process = news['text'].tolist()

#Start from the processed_count
for i in tqdm(range(processed_count, len(texts_to_process), BATCH_SIZE)):
    batch_texts = texts_to_process[i : i + BATCH_SIZE]

    #Run your batched pipeline
    batch_results = run_pipeline_batched(batch_texts)

    #Prepare small temporary dataframe for this batch
    batch_df = pd.DataFrame({
        'text': batch_texts,
        'propaganda': [json.dumps(res) for res in batch_results]
    })

    #Append to CSV: header=False because the file already has a header
    batch_df.to_csv(output_file, mode='a', index=False, header=False)

print(f"Processing complete. Results saved to {output_file}")

No cache found. Preparing dataset...


  0%|          | 0/2214 [00:00<?, ?it/s]

In [ ]:
#Load `news_with_labels.csv` if it already exists; otherwise, run the labeling pipeline and save the result
if output_file.exists():
    news = pd.read_csv(output_file)
    news["propaganda"] = news["propaganda"].apply(json.loads)
else:
    print("Cache not found. Downloading the data and running the RoBERTa labeling pipeline (this will take time)...")
    #Load the news article dataset
    news = load_dataset("vblagoje/cc_news", split="train")
    news = news.to_pandas()

    #Constrain to only the text, as that's the only input our extension will be given
    #And take a random sample of articles because the dataset is unreasonably large
    news = news[['text']].sample(frac=0.1, random_state=42).reset_index(drop=True)
    display(news)

    #Use run pipeline function to get predicted propaganda spans and labels from all the text
    print("Running pipeline...")
    BATCH_SIZE = 32
    all_predictions = []
    texts_to_process = news['text'].tolist()

    for i in tqdm(range(0, len(texts_to_process), BATCH_SIZE)):
        batch = texts_to_process[i : i + BATCH_SIZE]
        batch_results = run_pipeline_batched(batch)
        all_predictions.extend(batch_results)

    news['propaganda'] = all_predictions
    display(news)

    #Save the dataframe so it can be reused later
    print("Saving DataFrame...")
    news_to_save = news.copy()
    news_to_save["propaganda"] = news_to_save["propaganda"].apply(json.dumps)

    news_to_save.to_csv(output_file, index=False)
    print(f"Silver labels saved to {output_file}")

In [ ]:
import google.generativeai as genai

genai.configure(api_key="YOUR_API_KEY")
model = genai.GenerativeModel('gemini-1.5-flash')

def get_second_opinion(text, spans):
    prompt = f"""
    I am detecting propaganda. My base model found these potential spans: {spans}

    Article Text: "{text}"

    Here are the exact types of propaganda I'm looking for with their definitions and an example for each. Only look for and consider these specific techniques:
    1. Loaded language. Definition: Using specific words and phrases with strong emotional implications (either
    positive or negative) to influence an audience. Example: Outrage as Donald Trump suggests injecting disinfectant to kill virus.
    2. Name calling, labeling. Definition: Labeling the object of the propaganda campaign as either something the target audience fears, hates, finds undesirable or loves, praises. Example: Coronavirus emergency is ’Public Enemy Number 1’
    3. Repetition. Definition: Repeating the same message over and over again, so that the audience will eventually accept it. Example: I still have a dream. It is a dream deeply rooted in the American dream. I have a dream that one day
    4. Exaggeration, minimization. Definition: Either representing something in an excessive manner: making things larger, better, worse or making something seem less important or smaller than it actually is. Example: Coronavirus ‘risk to the American people remains very low’, Trump said.
    5. 

    Instructions:
    1. Review the spans found by the base model.
    2. If a span is actually neutral, remove it.
    3. If the base model missed a clear propaganda technique (like Loaded Language or Slogans), add it.
    4. Provide the final list in JSON format: [{"span": [start, end], "technique": "name"}]
    """
    response = model.generate_content(prompt)
    return response.text

In [ ]:
#Train/test/split
#Vectorize or otherwise transform text somehow
#Train models and compare performance
#Get some examples